# TP2 : descente de gradient, choix du pas et convergence
**Cours :** Algorithmes d'optimisation, séance 6 · **Intervenant :** Serigne Faye

**VERSION ENSEIGNANT (corrigé)**

## Objectifs
- implémenter la descente de gradient avec différentes règles de choix du pas ;
- étudier l'influence du pas fixe sur la convergence ;
- implémenter une recherche linéaire (backtracking) et le pas optimal ;
- comparer les critères d'arrêt et plusieurs configurations ;
- appliquer la descente de gradient à un problème d'apprentissage (régression linéaire).

## Consignes
Exécutez les cellules dans l'ordre, complétez les cellules marquées **À COMPLÉTER** et répondez aux questions dans les cellules de texte.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

## Partie 1 : la descente de gradient
On cherche à minimiser $f$ en itérant $x_{k+1} = x_k - t_k \nabla f(x_k)$. La fonction `gradient_descent` ci-dessous accepte un pas fixe (un nombre) ou une **règle de pas** (une fonction `regle(x, g)` renvoyant le pas $t_k$). Elle propose trois critères d'arrêt :
- `"grad"` : $\|\nabla f(x_k)\| < \text{tol}$ ;
- `"pas"` : $\|x_{k+1} - x_k\| < \text{tol}$ ;
- `"fval"` : $|f(x_{k+1}) - f(x_k)| < \text{tol}$.

**Exercice 1.1.** Compléter les deux lignes marquées.

In [ ]:
def gradient_descent(f, grad, x0, step, max_iter=10000, tol=1e-6, crit="grad"):
    x = np.array(x0, dtype=float)
    xs, fs, gn, ts = [x.copy()], [f(x)], [np.linalg.norm(grad(x))], []
    converged = False
    for k in range(max_iter):
        g = grad(x)
        if crit == "grad" and np.linalg.norm(g) < tol:
            converged = True; break
        t = step(x, g) if callable(step) else step
        x_new = x - t * g
        if not np.all(np.isfinite(x_new)) or np.linalg.norm(x_new) > 1e8:
            break                        # divergence
        small_step = np.linalg.norm(x_new - x) < tol
        small_f = abs(f(x_new) - f(x)) < tol
        x = x_new
        xs.append(x.copy()); fs.append(f(x)); gn.append(np.linalg.norm(grad(x))); ts.append(t)
        if (crit == "pas" and small_step) or (crit == "fval" and small_f):
            converged = True; break
    return x, {"xs": np.array(xs), "fs": np.array(fs), "gn": np.array(gn), "ts": np.array(ts),
               "iters": len(xs) - 1, "converged": converged}

On teste sur $f(x, y) = x^2 + 10y^2$, de gradient $(2x, 20y)^\top$, en partant de $(1, 1)$.

In [ ]:
def f(p): return p[0]**2 + 10*p[1]**2
def grad(p): return np.array([2*p[0], 20*p[1]])
x0 = np.array([1.0, 1.0])

x, h = gradient_descent(f, grad, x0, step=0.05)
print(x, h["iters"], h["converged"])   # attendu : proche de (0, 0)

## Partie 2 : influence d'un pas fixe
**Exercice 2.1.** Pour chaque pas de la liste, lancer la descente (`max_iter=2000`) et afficher le nombre d'itérations, si l'algorithme a convergé et la valeur finale de $f$. Tracer ensuite $f(x_k)$ en échelle logarithmique pour chaque pas.

In [ ]:
pas_liste = [0.001, 0.01, 0.05, 0.09, 0.1, 0.11]
resultats = {}
for t in pas_liste:
    _, resultats[t] = gradient_descent(f, grad, x0, step=t, max_iter=2000)

print("   pas | itérations | convergé |   f finale")
for t in pas_liste:
    r = resultats[t]
    print(f"{t:6.3f} | {r['iters']:10d} | {str(r['converged']):8s} | {r['fs'][-1]:11.3e}")

plt.figure(figsize=(8, 5))
for t in pas_liste:
    plt.semilogy(resultats[t]["fs"], label=f"pas = {t}")
plt.xlabel("itération"); plt.ylabel("f(x_k)"); plt.legend(); plt.grid(True); plt.show()

**Question 2.2.** Classez les pas : lequel est le plus rapide ? lequel est trop lent ? lequel diverge ? Que se passe-t-il pour $t = 0{,}1$ ? Comparez avec la condition $t < 2/L$ où $L = 20$ est la plus grande courbure de $f$ (valeur propre maximale de la hessienne $\mathrm{diag}(2, 20)$).

**Réponse :** $t = 0{,}09$ est le plus rapide (proche du pas optimal $2/(\mu + L) \approx 0{,}09$) ; $t = 0{,}001$ et $0{,}01$ sont trop lents ; $t = 0{,}11$ diverge. Pour $t = 0{,}1 = 2/L$ (valeur critique), la composante en $y$ est multipliée par $1 - 20t = -1$ : elle oscille indéfiniment sans converger. La condition $t < 2/L = 0{,}1$ est bien celle observée.

## Partie 3 : recherche linéaire (backtracking)
Plutôt que de fixer $t$, on le choisit à chaque itération : on part de $t = 1$ et on le divise par 2 tant que la condition d'Armijo n'est pas satisfaite :
$$f(x - t g) \leq f(x) - c\, t\, \|g\|^2, \qquad g = \nabla f(x), \quad c = 10^{-4}.$$

**Exercice 3.1.** Écrire `backtracking(f, grad, x, t0, beta, c)` puis lancer la descente avec la règle `lambda x, g: backtracking(f, grad, x)`.

In [ ]:
def backtracking(f, grad, x, t0=1.0, beta=0.5, c=1e-4):
    g = grad(x); t = t0
    while f(x - t*g) > f(x) - c * t * (g @ g):
        t *= beta
    return t

x_bt, h_bt = gradient_descent(f, grad, x0, step=lambda x, g: backtracking(f, grad, x))
print(h_bt["iters"], h_bt["converged"], x_bt)

**Question 3.2.** Comparez le nombre d'itérations avec le meilleur pas fixe de la partie 2. Quel est l'avantage du backtracking ? Quel est son coût ?

**Réponse :** Le backtracking converge sans qu'on ait à régler le pas (il s'adapte à la fonction et ne diverge pas), pour un nombre d'itérations comparable (voire meilleur) à celui du meilleur pas fixe (environ 60 contre 77 pour $t = 0{,}09$), mais il évalue $f$ plusieurs fois par itération (coût supplémentaire).

## Partie 4 : pas optimal pour une fonction quadratique
Pour $f(x) = \tfrac12 x^\top A x$ avec $A = \mathrm{diag}(2, 20)$ (c'est notre fonction $x^2 + 10y^2$), la recherche linéaire exacte dans la direction $-g$ donne le pas
$$t_k = \frac{g^\top g}{g^\top A g}.$$

**Exercice 4.1.** Implémenter cette règle, lancer la descente depuis le point $(1\,;\,0{,}1)$ (choisi pour bien voir le comportement) et tracer sur les courbes de niveau les trajectoires obtenues avec le pas optimal et avec $t = 0{,}05$.

In [ ]:
A = np.diag([2.0, 20.0])
regle_optimale = lambda x, g: (g @ g) / (g @ A @ g)
x0b = np.array([1.0, 0.1])

x_opt, h_opt = gradient_descent(f, grad, x0b, step=regle_optimale)
x_fix, h_fix = gradient_descent(f, grad, x0b, step=0.05)
print("pas optimal :", h_opt["iters"], "itérations ; pas 0.05 :", h_fix["iters"], "itérations")

xs = np.linspace(-1.2, 1.2, 200); X, Y = np.meshgrid(xs, xs)
plt.figure(figsize=(6, 6)); plt.contour(X, Y, X**2 + 10*Y**2, levels=20, alpha=0.4)
plt.plot(h_opt["xs"][:, 0], h_opt["xs"][:, 1], "o-", ms=3, label="pas optimal")
plt.plot(h_fix["xs"][:, 0], h_fix["xs"][:, 1], "s-", ms=3, label="pas 0.05")
plt.legend(); plt.show()

**Question 4.2.** Décrivez les trajectoires. Pourquoi « zigzaguent-elles » dans une vallée étroite ?

**Réponse :** Avec le pas optimal, chaque direction de descente est orthogonale à la précédente (on minimise exactement sur chaque droite), d'où un zigzag entre les deux flancs de la vallée (ici 75 itérations pour le pas optimal contre 138 pour $t = 0{,}05$). Plus la vallée est étroite (courbures très différentes), plus le zigzag est marqué et la convergence lente.

## Partie 5 : influence du conditionnement
On considère $f_\kappa(x, y) = \tfrac12 (x^2 + \kappa y^2)$, de hessienne $\mathrm{diag}(1, \kappa)$. Le **conditionnement** est $\kappa$.

**Exercice 5.1.** Pour $\kappa \in \{1, 10, 100\}$, en partant de $(1\,;\,1/\kappa)$, comparer le nombre d'itérations (critère `"grad"`, `tol=1e-6`) avec : un pas fixe $2/(1+\kappa)$, le backtracking et le pas optimal.

In [ ]:
def quadratique(kappa):
    A = np.diag([1.0, kappa])
    return (lambda p: 0.5 * p @ A @ p), (lambda p: A @ p), A

print(" kappa | pas fixe | backtracking | pas optimal")
for kappa in [1, 10, 100]:
    fk, gk, Ak = quadratique(kappa)
    xk = np.array([1.0, 1.0 / kappa])
    _, h1 = gradient_descent(fk, gk, xk, step=2/(1 + kappa), max_iter=20000)
    _, h2 = gradient_descent(fk, gk, xk, step=lambda x, g: backtracking(fk, gk, x), max_iter=20000)
    _, h3 = gradient_descent(fk, gk, xk, step=lambda x, g: (g @ g) / (g @ Ak @ g), max_iter=20000)
    print(f"{kappa:6d} | {h1['iters']:8d} | {h2['iters']:12d} | {h3['iters']:11d}")

**Question 5.2.** Comment le nombre d'itérations évolue-t-il avec $\kappa$ ? Pourquoi un mauvais conditionnement est-il un problème ?

**Réponse :** Le nombre d'itérations croît à peu près proportionnellement à $\kappa$ (environ 70 pour $\kappa = 10$ et 700 pour $\kappa = 100$ avec ce départ), pour les trois méthodes (vitesse linéaire de facteur $(\kappa - 1)/(\kappa + 1)$). Remarque pour l'enseignant : depuis un départ particulier comme $(1, 1)$, le pas optimal peut converger en quelques itérations, d'où le choix du départ $(1, 1/\kappa)$. Une vallée très étroite oblige à utiliser un petit pas à cause de la direction de forte courbure, ce qui rend la progression lente dans la direction de faible courbure.

## Partie 6 : une fonction difficile (Rosenbrock)
$f(x, y) = (1 - x)^2 + 100 (y - x^2)^2$, de minimum $(1, 1)$, à minimiser depuis $(-1{,}2\, ;\, 1)$.

**Exercice 6.1.** Comparer : pas fixe $10^{-3}$ (20 000 itérations maximum), backtracking (50 000 itérations maximum, `tol=1e-6`) et `scipy.optimize.minimize` avec la méthode `"BFGS"` (méthode quasi-Newton, vue à la séance suivante).

In [ ]:
rosen = lambda p: (1 - p[0])**2 + 100*(p[1] - p[0]**2)**2
rgrad = lambda p: np.array([-2*(1 - p[0]) - 400*p[0]*(p[1] - p[0]**2), 200*(p[1] - p[0]**2)])
r0 = np.array([-1.2, 1.0])

xa, ha = gradient_descent(rosen, rgrad, r0, step=1e-3, max_iter=20000)
xb, hb = gradient_descent(rosen, rgrad, r0, step=lambda x, g: backtracking(rosen, rgrad, x), max_iter=50000)
res = minimize(rosen, r0, jac=rgrad, method="BFGS")
print("pas fixe 1e-3 :", xa, ha["iters"], "itérations, convergé :", ha["converged"])
print("backtracking  :", xb, hb["iters"], "itérations, convergé :", hb["converged"])
print("BFGS (scipy)  :", res.x, res.nit, "itérations")

**Question 6.2.** Classez les trois méthodes. Que laisse penser le résultat de BFGS sur l'intérêt d'utiliser de l'information du second ordre ?

**Réponse :** BFGS (quelques dizaines d'itérations) est de loin le plus efficace, le backtracking demande plusieurs milliers d'itérations et le pas fixe $10^{-3}$ n'a pas convergé au bout de 20 000 itérations. Exploiter la courbure (hessienne exacte pour Newton, approchée pour BFGS) permet de corriger la direction de descente dans une vallée étroite : c'est le sujet de la séance 7.

## Partie 7 : application à l'apprentissage (régression linéaire)
On dispose de $N = 50$ points $(x_i, y_i)$ et on cherche la droite $y = a x + b$ qui minimise l'erreur quadratique moyenne
$$L(a, b) = \frac{1}{N} \sum_{i=1}^{N} (a x_i + b - y_i)^2 .$$

**Exercice 7.1.** Écrire `L(p)` et `grad_L(p)` pour $p = (a, b)$, avec $\nabla L = \left(\dfrac{2}{N}\sum (a x_i + b - y_i) x_i,\; \dfrac{2}{N}\sum (a x_i + b - y_i)\right)^\top$. Lancer la descente de gradient avec un pas fixe $0{,}1$ depuis $(0, 0)$ et comparer avec `np.polyfit`.

In [ ]:
rng = np.random.default_rng(0)
xd = np.linspace(0, 1, 50)
yd = 2*xd + 1 + 0.1*rng.normal(size=50)

def L(p):
    return np.mean((p[0]*xd + p[1] - yd)**2)
def grad_L(p):
    r = p[0]*xd + p[1] - yd
    return np.array([2*np.mean(r*xd), 2*np.mean(r)])

p_gd, hL = gradient_descent(L, grad_L, np.zeros(2), step=0.1, max_iter=5000, tol=1e-8)
print("descente de gradient :", p_gd, hL["iters"], "itérations")
print("np.polyfit           :", np.polyfit(xd, yd, 1))

fig, axs = plt.subplots(1, 2, figsize=(11, 4))
axs[0].plot(xd, yd, "o", ms=3); axs[0].plot(xd, p_gd[0]*xd + p_gd[1], "r-"); axs[0].set_title("Données et droite ajustée")
axs[1].semilogy(hL["fs"]); axs[1].set_title("Perte L en fonction de l'itération"); axs[1].set_xlabel("itération")
plt.show()

**Exercice 7.2.** Refaire la descente avec les pas $0{,}01$, $0{,}5$ et $0{,}9$ (`max_iter=3000`). Pour chacun, indiquer si l'algorithme converge et combien d'itérations sont utilisées.

In [ ]:
for t in [0.01, 0.5, 0.9]:
    pt, ht = gradient_descent(L, grad_L, np.zeros(2), step=t, max_iter=3000, tol=1e-8)
    print(f"pas = {t:4.2f} : {ht['iters']:5d} itérations, convergé = {ht['converged']}, (a, b) = {pt}")

**Question 7.3.** Quel rôle joue le pas, appelé ici *learning rate* en apprentissage automatique ? Que se passe-t-il s'il est trop petit, trop grand ?

**Réponse :** Le pas règle la vitesse d'apprentissage. Trop petit ($0{,}01$) : très lent (3000 itérations ne suffisent pas à converger). Raisonnable ($0{,}1$ à $0{,}5$) : convergence vers la même solution que `polyfit`. Trop grand ($0{,}9$) : la perte diverge, car le pas dépasse $2/L$ où $L$ est la plus grande valeur propre de la hessienne de $L$ (environ $2{,}5$ ici, donc un pas critique d'environ $0{,}8$).

## Partie 8 : synthèse
**Question 8.1.** Pour chacun des critères d'arrêt `"grad"`, `"pas"` et `"fval"`, donnez un avantage et une limite. Lequel choisiriez-vous en pratique, et avec quelle sécurité supplémentaire ?

**Réponse :** `grad` : directement lié à la condition d'optimalité $\nabla f = 0$, mais l'échelle de $\nabla f$ dépend de la fonction. `pas` : facile à interpréter (« on ne bouge plus »), mais un petit pas peut aussi venir d'un pas $t$ minuscule sans que l'on soit près du minimum. `fval` : indique que la fonction ne baisse plus, mais peut être trompeur sur un plateau. En pratique, on combine le critère sur la norme du gradient avec un **nombre maximal d'itérations** pour éviter les boucles infinies.

**Question 8.2.** Résumez en 5 lignes comment choisir entre pas fixe, backtracking et pas optimal.

**Réponse :** Pas fixe : simple et peu coûteux, mais il faut connaître (ou deviner) $L$ et un mauvais choix ralentit ou fait diverger. Backtracking : robuste et général, un peu plus coûteux par itération. Pas optimal (recherche linéaire exacte) : intéressant surtout pour les fonctions quadratiques. Dans tous les cas, un mauvais conditionnement ralentit la descente de gradient ; les méthodes de Newton et quasi-Newton (séance 7) y remédient.